# Polars Bokeh Accessor Example

Minimal example using the `pl.DataFrame.bokeh.line_plot` accessor.


Minimal example showing how to use the Polars Bokeh accessor to build charts.

**What this notebook does**
- Builds a figure and layers multiple calls on the same plot.
- Demonstrates Polars + Bokeh interoperability.

**Requirements**
- `polars`
- `bokeh`
- `pandas` (for conversion helpers)


In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import polars as pl
import datetime as dt

from bokeh.io import output_notebook
from bokeh.plotting import show

# Ensure accessor is registered
from spectral.charts import polars_charts  # noqa: F401
from spectral.charts.theme_manager import theme

output_notebook()
theme.set("dark_minimal")


Loading BokehJS ...

## Simple pandas `.plot` Example

Baseline pandas plotting example for quick comparison.


In [ ]:
pandas_df = pd.DataFrame({"x": [1, 2, 3, 4], "y": [2, 1, 3, 5], "z": [1, 2, 3, 2]})
pandas_df.plot(x="x", y=["y", "z"], kind="line", title="Simple pandas plot", label=["A", "B"])


## Multiple Calls on the Same Figure

Reuse a single Bokeh figure to mimic `df.plot(ax=...)` behavior.


In [ ]:
from bokeh.plotting import figure, show
from bokeh.layouts import row, column

df1 = pl.DataFrame({'A': [1, 2, 3, 4]})
df2 = pl.DataFrame({'B': [2, 1, 3, 2]})
df3 = pl.DataFrame({'C1': [0.0, 0.1, 0.5, 1.5, 2], 'C2': [1, 1, 2, 3, 2]})

fig1 = df1.bokeh.line(y='A', line_color="blue", line_alpha=0.6, title="Multiple df.bokeh.line_plot() Calls on Same Figure")
df2.bokeh.line(y="B", figure=fig1)
df3.bokeh.line(y=["C1", "C2"], title="override", line_color="green", figure=fig1)

show(fig1);

## Renderer-Level Kwargs

These are properties on Bokeh's `GlyphRenderer` (not the glyph model itself).
They are forwarded via `df.bokeh.line(...)` and applied to the renderer.


In [ ]:
df = pl.DataFrame({"x": [1, 2, 3], "y": [2, 5, 3]})

fig = df.bokeh.line(
    x="x",
    y="y",
    line_color="tomato",  # glyph property
    visible=True,          # renderer property
    muted=True,            # renderer property
    level="overlay",     # renderer property
    name="series_a",     # renderer property
    tags=["primary"],    # renderer property
)

show(fig);


In [5]:
import bokeh
bokeh.models.Line.properties()

{'decorations',
 'js_event_callbacks',
 'js_property_callbacks',
 'line_alpha',
 'line_cap',
 'line_color',
 'line_dash',
 'line_dash_offset',
 'line_join',
 'line_width',
 'name',
 'subscribed_events',
 'syncable',
 'tags',
 'x',
 'y'}

## Multiple Hover Tools (Per-Glyph)

Attach different hover tools to different glyphs on the same figure.


In [10]:
from bokeh.models import HoverTool

df = pl.DataFrame({
    "x": [1, 2, 3, 4],
    "y1": [2, 5, 3, 6],
    "y2": [1, 2, 4, 3],
})

fig = df.bokeh.line(
    x="x",
    y="y1",
    line_color="tomato",
    line_name="series_1",
)
df.bokeh.line(
    x="x",
    y="y2",
    figure=fig,
    line_color="steelblue",
    line_name="series_2",
)

r1 = fig.select_one({"name": "series_1"})
r2 = fig.select_one({"name": "series_2"})

hover1 = HoverTool(tooltips=[("y1 parara", "@y1")], renderers=[r1])
hover2 = HoverTool(tooltips=[("y2", "@y2")], renderers=[r2])
fig.add_tools(hover1, hover2)

show(fig);


In [6]:
figure.properties()

{'above',
 'align',
 'aspect_ratio',
 'aspect_scale',
 'attribution',
 'background_fill_alpha',
 'background_fill_color',
 'background_hatch_alpha',
 'background_hatch_color',
 'background_hatch_extra',
 'background_hatch_pattern',
 'background_hatch_scale',
 'background_hatch_weight',
 'below',
 'border_fill_alpha',
 'border_fill_color',
 'border_hatch_alpha',
 'border_hatch_color',
 'border_hatch_extra',
 'border_hatch_pattern',
 'border_hatch_scale',
 'border_hatch_weight',
 'border_line_alpha',
 'border_line_cap',
 'border_line_color',
 'border_line_dash',
 'border_line_dash_offset',
 'border_line_join',
 'border_line_width',
 'center',
 'context_menu',
 'css_classes',
 'css_variables',
 'disabled',
 'elements',
 'extra_x_ranges',
 'extra_x_scales',
 'extra_y_ranges',
 'extra_y_scales',
 'flow_mode',
 'frame_align',
 'frame_height',
 'frame_width',
 'height',
 'height_policy',
 'hidpi',
 'hold_render',
 'html_attributes',
 'html_id',
 'inner_height',
 'inner_width',
 'js_event_call

In [ ]:
from bokeh.models import Legend, LegendItem

Legend.properties()

In [ ]:
Legend.properties()

In [ ]:
from bokeh.models.renderers import GlyphRenderer